In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

# All inputs needed for model: date, close price pairs of stock data

# ==========================================
# 1. DATA LOADING & FEATURE ENGINEERING
# ==========================================
data = pd.read_csv("data/MicrosoftStock.csv")
data['date'] = pd.to_datetime(data['date'])

data['month_sin'] = np.sin(2 * np.pi * data['date'].dt.month / 12)
data['month_cos'] = np.cos(2 * np.pi * data['date'].dt.month / 12)
data['day_sin'] = np.sin(2 * np.pi * data['date'].dt.dayofweek / 7)
data['day_cos'] = np.cos(2 * np.pi * data['date'].dt.dayofweek / 7)

data['vol_pct_change'] = data['volume'].pct_change()

# Using % change (Returns) to make data stationary (prevents trend-following overfitting)
data['return'] = data['close'].pct_change()

# IMPROVEMENT: Add a 5-day Moving Average of returns to give the model "momentum" context
data['return_mavg'] = data['return'].rolling(window=5).mean()

# Drop rows with NaN values created by pct_change and rolling mean
data = data.dropna().reset_index(drop=True)

feature_cols = ['return', 'return_mavg', 'vol_pct_change', 'month_sin', 'month_cos', 'day_sin', 'day_cos']
dataset = data[feature_cols].values

print(dataset)

# ==========================================
# 2. DATA SPLITTING & SCALING
# ==========================================
training_data_len = int(np.ceil(len(dataset)) * 0.95)

# Scaler is fitted ONLY on training data to avoid data leakage from the future
scaler = StandardScaler()
scaler.fit(dataset[:training_data_len])

scaled_train = scaler.transform(dataset[:training_data_len])
# For testing, we need the 60 days prior to the test start for the first window
scaled_test_source = scaler.transform(dataset[training_data_len - 60:])

print(scaled_train)

[[-1.06990014e-03  3.32744100e-03  5.20076953e-01 ...  5.00000000e-01
  -4.33883739e-01 -9.00968868e-01]
 [ 1.24955373e-03  1.32689803e-03 -2.18445206e-01 ...  5.00000000e-01
   7.81831482e-01  6.23489802e-01]
 [-6.23997147e-03 -6.46712824e-05  1.36705283e-01 ...  5.00000000e-01
   9.74927912e-01 -2.22520934e-01]
 ...
 [-4.11854435e-02 -1.26952814e-02  6.60927619e-02 ...  5.00000000e-01
   0.00000000e+00  1.00000000e+00]
 [ 3.78409091e-02 -2.61432270e-03  3.32483087e-01 ...  5.00000000e-01
   7.81831482e-01  6.23489802e-01]
 [-1.88328041e-02 -1.12762900e-02 -3.95463822e-01 ...  5.00000000e-01
   9.74927912e-01 -2.22520934e-01]]
[[-0.1470208   0.36889193  0.88613717 ...  0.80652752 -1.54224599
  -1.05128013]
 [ 0.01551673  0.04815769 -0.5823096  ...  0.80652752  0.81671816
   0.92310078]
 [-0.50931756 -0.1749437   0.12385687 ...  0.80652752  1.19140094
  -0.17259796]
 ...
 [ 0.66089299  1.98597847  0.00228072 ...  1.33399651  0.14156313
  -1.05128013]
 [ 0.00298981 -0.03563954 -0.674982